###Alphafold2 Conformation Generation Pipeline
Notebook and code written by Felipe Engelberger, 2024.
Built using ColabDesign by Sergey O.

felipeengelberger@gmail.com

In [ ]:
%%time
#%pip uninstall chex -y
# @title setup {"display-mode":"form"}
%pip -qq install git+https://github.com/sokrypton/ColabDesign.git@gamma
# @title Install
%cd /content
%pip install -q biopython igraph leidenalg
%pip -q install git+https://github.com/engelberger/frustrapy.git@dev
%pip install -q -U kaleido==0.2.1
%cd /workspaces/alphamask/
%pip install -e .


In [ ]:
# @title AlphaFold2 Masking-Mutate Experiment {"display-mode":"form"}

# @markdown # Generalized AlphaFold2 Masking Experiment

# @markdown This script provides a flexible framework for running iterative masking experiments with AlphaFold2.
# @markdown By default, it performs single-position masking across your sequence. You can either:
# @markdown - Let it iterate through the entire sequence automatically
# @markdown - Provide a specific list of positions to iterate through
# @markdown - Alternatively, use traditional masked/unmasked position lists
# @markdown See the FAQ section at the bottom of this notebook for detailed usage examples.

from alphamask.utils import run_masking_experiment

unified_memory = False #@param {type:"boolean"}

# @markdown ## Input Parameters

# @markdown **Input Sequence**
sequence = "GSHMASMEDLQAEARAFLSEEMIAEFKAAFDMFDADGGGDISYKAVGTVFRMLGINPSKEVLDYLKEKIDVDGSGTIDFEEFLVLMVYIMKQDA"  # @param {type:"string"}

# @markdown **Job Name Prefix**
jobname_prefix = "I89"  # @param {type:"string"}

# @markdown **Masking Strategy** (see FAQ for detailed explanation)
masking_strategy = "iterative_single"  # @param ["iterative_single","iterative_single_mask_mutate", "mask_positions", "unmask_positions"]

# @markdown **Positions to Process** (leave empty for full sequence in iterative mode, or enter comma-separated positions)
positions_str = ""  # @param {type:"string"}

# @markdown **Parent Path** (directory where results will be saved)
parent_path = "/content"  # @param {type:"string"}

# @markdown **AlphaFold2 Parameters**
num_recycles = 1  # @param {type:"slider", min:1, max:20, step:1}
num_seeds = 1  # @param {type:"slider", min:1, max:20, step:1}

# @markdown **MSA Method**
msa_method = "mmseqs2"  # @param ["mmseqs2","single_sequence", "custom_fas", "custom_a3m", "custom_sto"]

# @markdown **Custom a3m path** (leave empty for upload dialog, ignored for mmseqs2)
custom_a3m_path = "/here/your_aln.a3m"  # @param {type:"string"}

# @markdown **Run Control (Vanilla AF2)**
run_control = True  # @param {type:"boolean"}

# @markdown **Run Only Control**
run_only_control = True  # @param {type:"boolean"}

# @markdown **Mutations to Process** (leave empty for full sequence in iterative mode, or enter comma-separated positions)
mutations = ""  # @param {type:"string"}

# @markdown **Use Unified Memory**
unified_memory = False  # @param {type:"boolean"}

# Run the experiment
vanilla_control_jobname, vanilla_control_path = run_masking_experiment(
    sequence=sequence,
    jobname_prefix=jobname_prefix,
    parent_path=parent_path,
    masking_strategy=masking_strategy,
    positions_str=positions_str,
    num_recycles=num_recycles,
    num_seeds=num_seeds,
    msa_method=msa_method,
    custom_a3m_path=custom_a3m_path,
    mutations=mutations,
    run_control=run_control,
    run_only_control=run_only_control,
    unified_memory=unified_memory
)

# Configurational Frustration

In [ ]:
from os.path import exists, join
import os
import glob
import shutil
import sys
import frustrapy


# @title Frustratometer in Python {"display-mode":"form"}
mode = "configurational" # @param ["configurational", "singleresidue", "mutational"]
overwrite = True # @param {type:"boolean"}
debug = "INFO" # @param ["INFO","DEBUG"]


results_dir = f"/content/{vanilla_control_jobname}/{mode}"
pdbs_dir = f"{results_dir}/best_pdb/"

# First, check if source files exist
source_files = glob.glob(f"/content/{vanilla_control_jobname}/out/pdbs/*best*")
if not source_files:
    raise FileNotFoundError(f"No files matching '*best*' pattern found in /content/{jobname}/out/pdbs/")

# If overwrite is True, clear the results directory before creating new directories
if overwrite and exists(results_dir):
    os.system(f"rm -rf {results_dir}/*")

# Create the pdbs_dir
os.makedirs(pdbs_dir, exist_ok=True)

# Copy the best PDB file
assert len(source_files) == 1, f"Expected 1 best PDB file, found {len(source_files)}"
source_pdb = source_files[0]
pdb_file = os.path.basename(source_pdb)
destination_path = join(pdbs_dir, pdb_file)
shutil.copy(source_pdb, destination_path)

# Verify the file was copied correctly
if not exists(destination_path):
    raise FileNotFoundError(f"Failed to copy PDB file to {destination_path}")

# Now calculate frustration for the specific PDB file
try:
    pdb_config, plots_config, density_results, _ = frustrapy.calculate_frustration(
        pdb_file=destination_path,
        mode=mode,
        results_dir=results_dir,
        debug=debug.upper(),
        chain="A",
    )
    print("Frustration calculation completed successfully")
    print(f"Results saved in: {results_dir}")

except Exception as e:
    print(f"Error during frustration calculation: {str(e)}")
    # Print more detailed debugging information
    print(f"PDB file path: {destination_path}")
    print(f"PDB file exists: {exists(destination_path)}")
    print(f"Results directory exists: {exists(results_dir)}")
    raise

# You can add code here to work with the results
# pdb_config contains the PDB configuration
# plots_config contains the plotting configuration
# density_results contains the density calculation results

In [ ]:
# @title Select Figure to Display 📊 {"run":"auto","display-mode":"form"}
from IPython.display import clear_output
import plotly.io as pio
pio.renderers.default = 'vscode'

figure_name = "plot_5andens" # @param ["plot_5andens", "plot_5adens_proportions", "plot_contact_map"] {allow-input: true}

# Clear previous output
clear_output(wait=True)

# Display selected figure
if figure_name in plots_config:
    print(f"Displaying: {figure_name}")
    plots_config[figure_name].show()
else:
    print(f"Figure '{figure_name}' not found in plots_config!")
    print(f"Available figures: {list(plots_config.keys())}")

In [ ]:
# @title View Frustration Data 📊 {"run": "auto", "display-mode":"form"}
from IPython.display import clear_output
from google.colab import data_table
import pandas as pd

# Enable interactive data table formatting
data_table.enable_dataframe_formatter()

# Create filtering options
show_residue_range = True  # @param {type:"boolean"}
min_residue = 1  # @param {type:"slider", min:1, max:500, step:1}
max_residue = 94  # @param {type:"slider", min:1, max:500, step:1}
sort_by = "Rel_Minimally_Frustrated"  # @param ["Residue", "Total_Density", "Highly_Frustrated", "Neutrally_Frustrated", "Minimally_Frustrated", "Rel_Highly_Frustrated", "Rel_Neutrally_Frustrated", "Rel_Minimally_Frustrated"]
ascending = False  # @param {type:"boolean"}
rows_per_page = "50"  # @param [5, 10, 15, 20, 50, 100]

# Clear previous output
clear_output(wait=True)

# Create a list of dictionaries from the density results
data = []
for density in density_results.densities:
    data.append({
        'Residue': density.residue_number,
        'Chain': density.chain_id,
        'Total_Density': density.total_density,
        'Highly_Frustrated': density.highly_frustrated,
        'Neutrally_Frustrated': density.neutrally_frustrated,
        'Minimally_Frustrated': density.minimally_frustrated,
        'Rel_Highly_Frustrated': f"{density.rel_highly_frustrated:.2%}",
        'Rel_Neutrally_Frustrated': f"{density.rel_neutrally_frustrated:.2%}",
        'Rel_Minimally_Frustrated': f"{density.rel_minimally_frustrated:.2%}"
    })

# Create DataFrame
df = pd.DataFrame(data)

# Apply residue range filter if enabled
if show_residue_range:
    df = df[(df['Residue'] >= min_residue) & (df['Residue'] <= max_residue)]

# Sort the dataframe
if sort_by in ['Rel_Highly_Frustrated', 'Rel_Neutrally_Frustrated', 'Rel_Minimally_Frustrated']:
    # Convert percentage strings to floats for sorting
    sort_values = df[sort_by].str.rstrip('%').astype(float)
    df = df.iloc[sort_values.argsort()[::-1 if not ascending else 1]]
else:
    df = df.sort_values(by=sort_by, ascending=ascending)

# Print summary statistics
print(f"Showing {len(df)} residues")
if show_residue_range:
    print(f"Filtered for residues {min_residue}-{max_residue}")
print(f"Sorted by {sort_by} ({'ascending' if ascending else 'descending'})")
print("\nSummary Statistics:")
print(f"Average Total Density: {df['Total_Density'].mean():.2f}")
print(f"Average Minimally Frustrated: {df['Minimally_Frustrated'].mean():.2f}")

# Display interactive table
data_table.DataTable(df,
                    include_index=False,
                    num_rows_per_page=rows_per_page)


In [ ]:
# @title Single Residue Analysis for Top N Residues from Selected Range 🧬 {"run": "auto", "display-mode":"form"}
from IPython.display import clear_output
import pandas as pd
import os
from collections import defaultdict

# Amino acid conversion dictionaries
THREE_TO_ONE = {
    'ALA': 'A', 'CYS': 'C', 'ASP': 'D', 'GLU': 'E',
    'PHE': 'F', 'GLY': 'G', 'HIS': 'H', 'ILE': 'I',
    'LYS': 'K', 'LEU': 'L', 'MET': 'M', 'ASN': 'N',
    'PRO': 'P', 'GLN': 'Q', 'ARG': 'R', 'SER': 'S',
    'THR': 'T', 'VAL': 'V', 'TRP': 'W', 'TYR': 'Y'
}

# Parameters
n_top_residues = 10  # @param {type:"slider", min:1, max:20, step:1}
mode = "singleresidue"  # @param ["singleresidue", "mutational"]
debug = "INFO"  # @param ["INFO", "DEBUG"]

# Clear previous output
clear_output(wait=True)

# Get the top N residues from the previously filtered and sorted DataFrame (df)
top_n = df.head(n_top_residues)

print(f"Analyzing top {n_top_residues} residues from range {min_residue}-{max_residue}:")
print(f"(Sorted by {sort_by} {'ascending' if ascending else 'descending'})")
print("\nSelected residues:")
print(top_n[['Residue', 'Chain', sort_by]].to_string(index=False))

# Prepare residues for analysis
residues_to_analyze = defaultdict(list)
for _, row in top_n.iterrows():
    residues_to_analyze[row['Chain']].append(int(row['Residue']))

# Convert defaultdict to regular dict for frustrapy
residues_to_analyze = dict(residues_to_analyze)

print(f"\nPreparing to analyze residues: {residues_to_analyze}")

# Calculate total mutations to process
total_mutations = sum(len(residues) for residues in residues_to_analyze.values()) * 20

print(f"\nTotal mutations to analyze: {total_mutations}")

try:
    # Find the PDB file in the pdbs_dir
    pdb_files = [f for f in os.listdir(pdbs_dir) if f.endswith('.pdb')]
    if not pdb_files:
        raise FileNotFoundError(f"No PDB files found in {pdbs_dir}")
    pdb_file = pdb_files[0]

    print(f"\nUsing PDB file: {pdb_file}")

    # Calculate frustration for single residues
    pdb_config, plots_config, density_results, single_residue_data = frustrapy.calculate_frustration(
        pdb_file=os.path.join(pdbs_dir, pdb_file),
        mode=mode,
        results_dir=results_dir,
        debug=debug.upper(),
        chain="A",
        residues=residues_to_analyze,
    )

    print("\nAnalysis completed successfully!")

    # Dictionary to store most frustrated mutations for each residue
    most_frustrated_mutations = {}

    # Analyze single residue data
    if single_residue_data and "A" in single_residue_data:
        print("\nMost Frustrated Mutations Analysis:")
        print("-" * 50)

        for res_num in residues_to_analyze["A"]:
            if res_num in single_residue_data["A"]:
                res_data = single_residue_data["A"][res_num]
                mutations = res_data.mutations

                # Convert three-letter code to one-letter code
                native_one_letter = THREE_TO_ONE.get(res_data.residue_name.upper(), 'X')

                # Find most frustrated mutation
                most_frustrated = min(mutations.items(), key=lambda x: x[1])

                # Store in dictionary
                most_frustrated_mutations[res_num] = {
                    'native': native_one_letter,
                    'mutation': most_frustrated[0],
                    'frustration_index': most_frustrated[1]
                }

                print(f"\nPosition {res_num} (Native: {native_one_letter} [from {res_data.residue_name}])")
                print(f"Most frustrated mutation: {native_one_letter} → {most_frustrated[0]} "
                      f"(Frustration Index: {most_frustrated[1]:.3f})")

                # Sort and display top 5 most frustrated mutations
                sorted_mutations = sorted(mutations.items(), key=lambda x: x[1])
                print("\nTop 5 most frustrated mutations:")
                for mut, score in sorted_mutations[:5]:
                    print(f"  {native_one_letter} → {mut}: {score:.3f}")

        print("\nMost Frustrated Mutations Summary:")
        print("-" * 50)
        for res_num, data in most_frustrated_mutations.items():
            mutation_str = f"{data['native']}{res_num}{data['mutation']}"
            print(f"Residue {res_num}: {mutation_str} (FI: {data['frustration_index']:.3f})")

    else:
        print("\nNo single residue data available in the results")

    # Display plots if available
    if plots_config:
        print("\nAvailable plots:")
        for plot_name, plot in plots_config.items():
            print(f"- {plot_name}")
            plot.show()

except Exception as e:
    print(f"Error during analysis: {str(e)}")
    print("\nDebug information:")
    print(f"Results directory: {results_dir}")
    print(f"PDBs directory: {pdbs_dir}")
    print(f"Residues to analyze: {residues_to_analyze}")
    raise

# Create DataFrame for interactive display
if 'most_frustrated_mutations' in locals():
    mutation_df = pd.DataFrame([
        {
            'Residue': res_num,
            'Native': data['native'],
            'Mutation': data['mutation'],
            'Mutation_String': f"{data['native']}{res_num}{data['mutation']}",
            'Frustration_Index': data['frustration_index']
        }
        for res_num, data in most_frustrated_mutations.items()
    ])



print("\nInteractive Mutations Table:")
data_table.DataTable(mutation_df, include_index=False, num_rows_per_page=20)


In [ ]:
# @title Run AlphaMask for Most Frustrated Mutations 🧬 {"run": "auto", "display-mode":"form"}

# @markdown ## AlphaMask Mutation Analysis
# @markdown This cell takes the most frustrated mutations identified from the previous analysis and runs them through AlphaMask.
# @markdown Each mutation will be processed independently using the ITERATIVE_SINGLE_MASK_MUTATE strategy.

from IPython.display import clear_output
import os
import gc
import jax
import pandas as pd
from datetime import datetime
from alphamask.core import MutateAndMaskingPipeline

# Basic parameters
jobname_prefix = "I89"  # @param {type:"string"}
parent_path = "/content"  # @param {type:"string"}

# @markdown ### AlphaFold2 Configuration
num_recycles = 6  # @param {type:"slider", min:1, max:20, step:1}
num_seeds = 6  # @param {type:"slider", min:1, max:20, step:1}

# @markdown ### MSA Options
msa_method = "mmseqs2"  # @param ["mmseqs2", "single_sequence", "custom_fas", "custom_a3m", "custom_sto"] {type:"string"}
custom_a3m_path = ""  # @param {type:"string"}

# Clear previous output
clear_output(wait=True)

print("🔍 Found Most Frustrated Mutations:")
print("-" * 50)
for res_num, data in most_frustrated_mutations.items():
    mutation_str = f"{data['native']}{res_num}{data['mutation']}"
    print(f"Position {res_num}: {data['native']} → {data['mutation']} (Frustration Index: {data['frustration_index']:.3f})")

# Create pipeline instances for each mutation
pipelines = []
for res_num, data in most_frustrated_mutations.items():
    mutation_str = f"{data['native']}{res_num}{data['mutation']}"
    
    # Create pipeline parameters
    params = {
        "sequence": sequence,
        "jobname": f"{jobname_prefix}_mut_{mutation_str}",
        "parentPath": parent_path,
        "setupPath": "/content/setup",
        "masking_mode": "list",
        "mask_msa": True,
        "mask_deletion_matrix": True,
        "cols": [res_num],
        "mask_identity": "X",
        "mutations": [mutation_str],
        "num_recycles": num_recycles,
        "num_seeds": num_seeds,
        "msa_method": msa_method,
        "custom_a3m_path": custom_a3m_path,
        "run_control": False,
        "run_only_control": False,
        # Add other required parameters with default values
        "copies": 1,
        "pair_mode": "unpaired_paired",
        "cov": 75,
        "id": 90,
        "qid": 0,
        "do_not_filter": False,
        "template_mode": "none",
        "pdb": "",
        "chain": "A",
        "rm_template_seq": False,
        "propagate_to_copies": True,
        "do_not_align": False,
        "model_type": "monomer (ptm)",
        "rank_by": "plddt",
        "debug": False,
        "use_initial_guess": False,
        "num_msa": 512,
        "num_extra_msa": 1024,
        "use_cluster_profile": True,
        "model": "all",
        "recycle_early_stop_tolerance": 0.0,
        "select_best_across_recycles": False,
        "use_mlm": False,
        "use_dropout": False,
        "seed": 0,
        "show_images": True,
        "unified_memory": False,
        "cols_range": [],
        "overwrite": False,
        "show_figures": True
    }
    
    pipeline = MutateAndMaskingPipeline(params=params)
    pipelines.append((mutation_str, pipeline))

print(f"\n🚀 Preparing to run {len(pipelines)} AlphaMask experiments")
print(f"📁 Results will be saved in: {parent_path}")
print(f"⚙️ Using {num_recycles} recycles and {num_seeds} seeds per prediction")
print(f"🧬 MSA method: {msa_method}")
if custom_a3m_path:
    print(f"📄 Using custom MSA from: {custom_a3m_path}")

proceed = input("\n⚠️ Ready to start processing mutations? (y/n): ")
if proceed.lower() != 'y':
    print("❌ Operation cancelled")
    raise SystemExit

# Run pipelines
results = {}
for mutation_str, pipeline in pipelines:
    print(f"\n🔄 Processing mutation: {mutation_str}")
    print("-" * 50)

    try:
        start_time = datetime.now()
        jobname = pipeline.run()
        duration = datetime.now() - start_time

        results[mutation_str] = {
            'status': 'completed',
            'jobname': jobname,
            'duration': duration
        }
        print(f"✅ Successfully completed {mutation_str} in {duration}")

    except Exception as e:
        print(f"❌ Error processing {mutation_str}: {str(e)}")
        results[mutation_str] = {
            'status': 'failed',
            'error': str(e)
        }

    # Clear memory between runs
    backend = jax.lib.xla_bridge.get_backend()
    for buf in backend.live_buffers():
        buf.delete()
    gc.collect()

# Print summary and create results table
print("\n📊 Experiment Summary:")
print("-" * 50)
successful = sum(1 for r in results.values() if r['status'] == 'completed')
failed = len(results) - successful

print(f"\n📈 Final Statistics:")
print(f"Total mutations processed: {len(results)}")
print(f"Successful: {successful}")
print(f"Failed: {failed}")

# Create results dataframe
results_df = pd.DataFrame([
    {
        'Mutation': mutation_str,
        'Status': result['status'],
        'Jobname': result.get('jobname', 'N/A'),
        'Duration': str(result.get('duration', 'N/A')),
        'Error': result.get('error', 'N/A')
    }
    for mutation_str, result in results.items()
])

# Display results
display(results_df)

# Return results dictionary
results

In [ ]:
import os
import re

def rename_jobs(parent_path):
  """Renames job folders based on a specific pattern.

  Args:
    parent_path: The path to the directory containing the job folders.
  """

  job_name_mapping = {
      'I89_mut_E80P_posE80P_897a1': 'S89_mut_D78E_posD78E_0a2cd',
      'I89_mut_M86K_posM86K_897a1': 'S89_mut_E80P_posE80P_0a2cd',
      'I89_mut_V49K_posV49K_897a1': 'S89_mut_E81Y_posE81Y_0a2cd',
      'I89_mut_F79E_posF79E_897a1': 'S89_mut_F79E_posF79E_0a2cd',
      'I89_mut_V84D_posV84D_897a1': 'S89_mut_I77D_posI77D_0a2cd',
      'I89_mut_F82D_posF82D_897a1': 'S89_mut_L83K_posL83K_0a2cd',
      'I89_mut_I77D_posI77D_897a1': 'S89_mut_L85D_posL85D_0a2cd',
      'I89_vanilla_897a1': 'S89_mut_M86K_posM86K_0a2cd',
      'I89_mut_L83K_posL83K_897a1': 'S89_mut_V49K_posV49K_0a2cd',
      'I89_mut_L85D_posL85D_897a1': 'S89_mut_V84D_posV84D_0a2cd',
      'I89_mut_V87K_posV87K_897a1': 'S89_mut_V87K_posV87K_0a2cd',
  }

  for old_name, new_name in job_name_mapping.items():
    old_path = os.path.join(parent_path, old_name)
    new_path = os.path.join(parent_path, new_name)

    if os.path.exists(old_path):
      os.rename(old_path, new_path)
      print(f"Renamed '{old_name}' to '{new_name}'")
    else:
      print(f"WARNING: '{old_name}' not found.")

# Example usage:
parent_path = '/content/path/to/your/jobs'  # Replace with the actual path
rename_jobs(parent_path)

In [ ]:
# @title RMSD Analysis and Visualization Tool 📊 {"run": "auto", "display-mode":"form"}

# @markdown ## Comprehensive RMSD Analysis
# @markdown Analyze and visualize RMSD distributions and landscapes for mutant structures
# @markdown ### Parameters:

from IPython.display import clear_output, display, HTML
import matplotlib.pyplot as plt
import jax.numpy as jnp
import numpy as np
import jax
from Bio import PDB
import os
from pathlib import Path
import glob
import seaborn as sns
import matplotlib.gridspec as gridspec
import matplotlib.colors as mcolors
from matplotlib.colors import LinearSegmentedColormap, LogNorm
from tqdm.notebook import tqdm
import pandas as pd
from google.colab import data_table
import gc
import plotly
import plotly.io as pio
pio.renderers.default = "colab"
# Analysis Parameters
# @markdown ### Reference Structures
reference_pdb1 = "/content/state1.pdb"  # @param {type:"string"}
reference_pdb2 = "/content/state2.pdb"  # @param {type:"string"}
control_path = "/content/I89_vanilla_897a1/out/pdbs"  # @param {type:"string"}
parent_path = "/content"  # @param {type:"string"}

# @markdown ### Region Selection
start_residue = 1  # @param {type:"number"}
end_residue = 94  # @param {type:"number"}
use_region = True  # @param {type:"boolean"}

# @markdown ### Atom Selection
atom_selection = "CA"  # @param ["CA", "backbone", "all"] {type:"string"}

# @markdown ### Visualization Options
show_violin = True  # @param {type:"boolean"}
show_landscapes = False  # @param {type:"boolean"}
show_scatterplot = True  # @param {type:"boolean"}
include_control = True  # @param {type:"boolean"}
plot_width = 15  # @param {type:"slider", min:8, max:20, step:1}
plot_height = 10  # @param {type:"slider", min:4, max:15, step:1}

# Color scheme for landscapes
colorscale_cesar = [
    [0, "rgb(255, 255, 255)"],
    [1/12, "rgb(106, 34, 254)"],
    [3/12, "rgb(15, 166, 239)"],
    [5/12, "rgb(80, 244, 204)"],
    [7/12, "rgb(175, 244, 151)"],
    [9/12, "rgb(255, 167, 89)"],
    [11/12, "rgb(255, 34, 17)"],
    [1, "rgb(255, 0, 0)"]
]

def convert_rgb_string_to_tuple(rgb_string: str) -> tuple:
    return tuple(float(x) / 255 for x in rgb_string.replace("rgb", "").replace("(", "").replace(")", "").split(","))


# Core RMSD Functions and Data Processing
def jnp_kabsch(a, b):
    """Kabsch algorithm using JAX."""
    u, s, vh = jnp.linalg.svd(a.T @ b, full_matrices=False)
    u = jnp.where(jnp.linalg.det(u @ vh) < 0, u.at[:,-1].set(-u[:,-1]), u)
    return u @ vh

def jnp_rmsd(true, pred):
    """Calculate RMSD using JAX."""
    p = true - true.mean(0, keepdims=True)
    q = pred - pred.mean(0, keepdims=True)
    p = p @ jnp_kabsch(p, q)
    return jnp.sqrt(jnp.square(p-q).sum(-1).mean())

jnp_rmsd_parallel = jax.jit(jax.vmap(jnp_rmsd, (None,0)))

def extract_coordinates(pdb_file, atom_selection="CA", start_res=None, end_res=None):
    """Extract coordinates from PDB file based on atom selection and residue range."""
    parser = PDB.PDBParser(QUIET=True)
    structure = parser.get_structure("protein", pdb_file)

    if atom_selection == "CA":
        atoms = [atom for atom in structure.get_atoms()
                if atom.get_name() == "CA" and
                (not use_region or (start_res <= atom.get_parent().id[1] <= end_res))]
    elif atom_selection == "backbone":
        atoms = [atom for atom in structure.get_atoms()
                if atom.get_name() in ["N", "CA", "C", "O"] and
                (not use_region or (start_res <= atom.get_parent().id[1] <= end_res))]
    else:  # all atoms
        atoms = [atom for atom in structure.get_atoms()
                if not use_region or (start_res <= atom.get_parent().id[1] <= end_res)]

    if not atoms:
        raise ValueError(f"No atoms found in specified region {start_res}-{end_res}")

    return np.array([atom.get_coord() for atom in atoms])

def find_model_pdbs(jobname):
    """Find all model PDB files excluding best models."""
    pdb_dir = Path(parent_path) / jobname / "out" / "pdbs"
    if not pdb_dir.exists():
        return []

    all_pdbs = list(pdb_dir.glob(f"{jobname}*pdb"))
    model_pdbs = [pdb for pdb in all_pdbs if "best_model" not in str(pdb)]
    return model_pdbs

def calculate_model_rmsds(model_path, ref_coords1, ref_coords2=None):
    """Calculate RMSD values for a set of model structures."""
    model_files = [f for f in os.listdir(model_path) if f.endswith('.pdb') and 'best_model' not in f]

    rmsd_data = {
        'ref1_rmsds': [],
        'ref2_rmsds': [] if ref_coords2 is not None else None,
        'model_names': []
    }

    for pdb_file in tqdm(model_files, desc="Processing models"):
        try:
            model_coords = extract_coordinates(
                os.path.join(model_path, pdb_file),
                atom_selection,
                start_residue,
                end_residue
            )

            rmsd1 = float(jnp_rmsd(jnp.array(ref_coords1), jnp.array(model_coords)))
            rmsd_data['ref1_rmsds'].append(rmsd1)

            if ref_coords2 is not None:
                rmsd2 = float(jnp_rmsd(jnp.array(ref_coords2), jnp.array(model_coords)))
                rmsd_data['ref2_rmsds'].append(rmsd2)

            rmsd_data['model_names'].append(pdb_file)

        except Exception as e:
            print(f"Warning: Could not process {pdb_file}: {str(e)}")

    return rmsd_data

def calculate_statistics(rmsd_data):
    """Calculate comprehensive statistics for RMSD data."""
    stats = {}

    for ref in ['ref1', 'ref2']:
        if f'{ref}_rmsds' in rmsd_data and rmsd_data[f'{ref}_rmsds']:
            rmsds = np.array(rmsd_data[f'{ref}_rmsds'])
            stats[ref] = {
                'count': len(rmsds),
                'min': np.min(rmsds),
                'max': np.max(rmsds),
                'mean': np.mean(rmsds),
                'median': np.median(rmsds),
                'std': np.std(rmsds),
                'q1': np.percentile(rmsds, 25),
                'q3': np.percentile(rmsds, 75),
                'iqr': np.percentile(rmsds, 75) - np.percentile(rmsds, 25)
            }

    return stats

# Visualization Functions
def plot_rmsd_landscape(rmsd_ref1, rmsd_ref2, title, show_plot=True):
    """Create 2D histogram landscape plot for RMSD values."""
    fig = plt.figure(figsize=(plot_width, plot_height))
    gs = gridspec.GridSpec(2, 2, width_ratios=[5, 1], height_ratios=[1, 5],
                          hspace=0.2, wspace=0.2)
    ax_main = plt.subplot(gs[1, 0])
    ax_histx = plt.subplot(gs[0, 0], sharex=ax_main)
    ax_histy = plt.subplot(gs[1, 1], sharey=ax_main)

    # Get common range for all plots
    max_rmsd = max(max(rmsd_ref1), max(rmsd_ref2))

    # Create 2D histogram
    H, xedges, yedges = np.histogram2d(rmsd_ref1, rmsd_ref2, bins=50,
                                      range=[[0, max_rmsd], [0, max_rmsd]])

    # Create custom colormap with white for zero counts
    vmin = 0.1
    vmax = H.max()

    im = ax_main.pcolormesh(xedges, yedges, H.T, norm=LogNorm(vmin=vmin, vmax=vmax),
                           cmap=cesar_cmap)

    cbar = fig.colorbar(im, ax=ax_histy)
    cbar.set_label('log(counts)', labelpad=10)
    cbar.ax.minorticks_off()

    # Create 1D histograms
    ax_histx.hist(rmsd_ref1, bins=80, range=(0, max_rmsd), density=True, alpha=1, color='black')
    ax_histy.hist(rmsd_ref2, bins=80, range=(0, max_rmsd), density=True,
                  orientation='horizontal', alpha=1, color='black')

    # Customize spines and labels
    ax_histx.spines['right'].set_visible(False)
    ax_histx.spines['top'].set_visible(False)
    ax_histx.spines['bottom'].set_visible(False)
    ax_histx.tick_params(axis="x", labelbottom=False)
    ax_histy.spines['top'].set_visible(False)
    ax_histy.spines['right'].set_visible(False)
    ax_histy.spines['left'].set_visible(False)
    ax_histy.tick_params(axis="y", labelleft=False)

    ax_main.set_xlabel(r"RMSD vs Reference 1 (Å)", labelpad=10)
    ax_main.set_ylabel(r"RMSD vs Reference 2 (Å)", labelpad=10)
    plt.suptitle(title, fontweight='bold', y=0.95)

    ax_histx.set_ylabel('Density', labelpad=10)
    ax_histy.set_xlabel('Density', labelpad=10)

    plt.tight_layout()

    if show_plot:
        plt.show()

    return fig

def plot_violin_distributions(rmsd_data, include_control=True):
    """Create violin plots for RMSD distributions."""
    fig, axes = plt.subplots(1, 2 if reference_pdb2 else 1,
                            figsize=(plot_width, plot_height))
    if not reference_pdb2:
        axes = [axes]

    # Prepare data
    data_ref1 = []
    data_ref2 = []

    # Add control data if available
    if include_control and 'control_rmsds' in locals():
        data_ref1.extend([('Control', rmsd) for rmsd in control_rmsds['ref1_rmsds']])
        if reference_pdb2:
            data_ref2.extend([('Control', rmsd) for rmsd in control_rmsds['ref2_rmsds']])

    # Add mutation data
    for mutation, data in rmsd_data.items():
        data_ref1.extend([(mutation, rmsd) for rmsd in data['ref1_rmsds']])
        if reference_pdb2:
            data_ref2.extend([(mutation, rmsd) for rmsd in data['ref2_rmsds']])

    # Create DataFrames
    df_ref1 = pd.DataFrame(data_ref1, columns=['Structure', 'RMSD'])

    # Plot Reference 1
    sns.violinplot(data=df_ref1, x='Structure', y='RMSD', ax=axes[0])
    title = f'RMSD Distribution vs Reference 1\n({atom_selection} atoms'
    if use_region:
        title += f', residues {start_residue}-{end_residue}'
    title += ')'
    axes[0].set_title(title)
    axes[0].set_ylabel('RMSD (Å)')
    axes[0].tick_params(axis='x', rotation=45)

    # Plot Reference 2 if provided
    if reference_pdb2:
        df_ref2 = pd.DataFrame(data_ref2, columns=['Structure', 'RMSD'])
        sns.violinplot(data=df_ref2, x='Structure', y='RMSD', ax=axes[1])
        title = f'RMSD Distribution vs Reference 2\n({atom_selection} atoms'
        if use_region:
            title += f', residues {start_residue}-{end_residue}'
        title += ')'
        axes[1].set_title(title)
        axes[1].set_ylabel('RMSD (Å)')
        axes[1].tick_params(axis='x', rotation=45)

    plt.tight_layout()
    return fig

def save_detailed_data(rmsd_data, control_rmsds=None, output_dir=None):
    """Save detailed RMSD data for all structures."""
    # Create DataFrame for all RMSD values
    detailed_data = []

    # Add control data if available
    if control_rmsds is not None:
        for i, (rmsd1, name) in enumerate(zip(control_rmsds['ref1_rmsds'], control_rmsds['model_names'])):
            data_point = {
                'Structure': 'Control',
                'Model': name,
                'RMSD_Ref1': rmsd1
            }
            if reference_pdb2:
                data_point['RMSD_Ref2'] = control_rmsds['ref2_rmsds'][i]
            detailed_data.append(data_point)

    # Add mutation data
    for mutation, data in rmsd_data.items():
        for i, (rmsd1, name) in enumerate(zip(data['ref1_rmsds'], data['model_names'])):
            data_point = {
                'Structure': mutation,
                'Model': name,
                'RMSD_Ref1': rmsd1
            }
            if reference_pdb2:
                data_point['RMSD_Ref2'] = data['ref2_rmsds'][i]
            detailed_data.append(data_point)

    # Save as CSV
    detailed_df = pd.DataFrame(detailed_data)
    detailed_df.to_csv(os.path.join(output_dir, 'detailed_rmsd_data.csv'), index=False)

    # Save as NPY
    np.save(os.path.join(output_dir, 'detailed_rmsd_data.npy'), {
        'control': control_rmsds,
        'mutations': rmsd_data,
        'parameters': {
            'atom_selection': atom_selection,
            'region': (start_residue, end_residue) if use_region else None,
            'references': [reference_pdb1, reference_pdb2 if reference_pdb2 else None]
        }
    })

    return detailed_df

def create_landscape_summary(rmsd_data, control_rmsds=None):
    """Create summary figure with all landscape plots."""
    n_plots = len(rmsd_data) + (1 if control_rmsds is not None else 0)
    ncols = min(3, n_plots)
    nrows = (n_plots + ncols - 1) // ncols

    fig = plt.figure(figsize=(8*ncols, 8*nrows))
    plt.suptitle('RMSD Landscapes', fontsize=16)

    # Create individual subplots
    for idx in range(nrows * ncols):
        plt.subplot(nrows, ncols, idx + 1)

    plot_idx = 1

    # Plot control first if available
    if control_rmsds is not None and len(control_rmsds['ref1_rmsds']) > 0:
        ax = plt.subplot(nrows, ncols, plot_idx)
        ref2_data = control_rmsds['ref2_rmsds'] if reference_pdb2 else control_rmsds['ref1_rmsds']

        H, xedges, yedges = np.histogram2d(
            control_rmsds['ref1_rmsds'],
            ref2_data,
            bins=50,
            range=[[0, max(control_rmsds['ref1_rmsds'])],
                   [0, max(ref2_data)]]
        )

        im = ax.pcolormesh(xedges, yedges, H.T, norm=LogNorm(vmin=0.1, vmax=H.max()),
                          cmap=cesar_cmap)
        ax.set_title('Control')
        plot_idx += 1

    # Plot mutations
    for mutation, data in rmsd_data.items():
        if len(data['ref1_rmsds']) > 0:
            ax = plt.subplot(nrows, ncols, plot_idx)
            ref2_data = data['ref2_rmsds'] if reference_pdb2 else data['ref1_rmsds']

            H, xedges, yedges = np.histogram2d(
                data['ref1_rmsds'],
                ref2_data,
                bins=50,
                range=[[0, max(data['ref1_rmsds'])],
                       [0, max(ref2_data)]]
            )

            im = ax.pcolormesh(xedges, yedges, H.T, norm=LogNorm(vmin=0.1, vmax=H.max()),
                              cmap=cesar_cmap)
            ax.set_title(mutation)
        plot_idx += 1

    # Add common colorbar
    plt.colorbar(im, ax=plt.gcf().axes, label='log(counts)')

    plt.tight_layout(rect=[0, 0.03, 1, 0.95])
    return fig

def create_summary_statistics(rmsd_data, control_rmsds=None):
    """Create comprehensive summary statistics DataFrame."""
    stats_data = []

    # Add control statistics if available
    if control_rmsds is not None:
        control_stats = calculate_statistics(control_rmsds)
        control_dict = {
            'Structure': 'Control',
            'Num_Models': len(control_rmsds['ref1_rmsds'])
        }
        # Add Ref1 statistics
        for k, v in control_stats['ref1'].items():
            control_dict[f'Ref1_{k}'] = v
        # Add Ref2 statistics if available
        if reference_pdb2:
            for k, v in control_stats['ref2'].items():
                control_dict[f'Ref2_{k}'] = v
        stats_data.append(control_dict)

    # Add mutation statistics
    for mutation, data in rmsd_data.items():
        mutation_stats = calculate_statistics(data)
        mutation_dict = {
            'Structure': mutation,
            'Num_Models': len(data['ref1_rmsds'])
        }
        # Add Ref1 statistics
        for k, v in mutation_stats['ref1'].items():
            mutation_dict[f'Ref1_{k}'] = v
        # Add Ref2 statistics if available
        if reference_pdb2 and 'ref2' in mutation_stats:
            for k, v in mutation_stats['ref2'].items():
                mutation_dict[f'Ref2_{k}'] = v
        stats_data.append(mutation_dict)

    return pd.DataFrame(stats_data)

def extract_plddt_from_pdb(pdb_file):
    """Extract pLDDT values (B-factor column) from PDB file."""
    parser = PDB.PDBParser(QUIET=True)
    structure = parser.get_structure("protein", pdb_file)

    plddt_values = []
    if atom_selection == "CA":
        atoms = [atom for atom in structure.get_atoms()
                if atom.get_name() == "CA" and
                (not use_region or (start_residue <= atom.get_parent().id[1] <= end_residue))]
    elif atom_selection == "backbone":
        atoms = [atom for atom in structure.get_atoms()
                if atom.get_name() in ["N", "CA", "C", "O"] and
                (not use_region or (start_residue <= atom.get_parent().id[1] <= end_residue))]
    else:
        atoms = [atom for atom in structure.get_atoms()
                if not use_region or (start_residue <= atom.get_parent().id[1] <= end_residue)]

    # Calculate mean pLDDT for the selected atoms
    plddt = np.mean([atom.get_bfactor() for atom in atoms])
    return plddt

def create_interactive_plots(rmsd_data, control_rmsds=None):
    """Create interactive Plotly scatter plots of RMSDs colored by pLDDT."""
    import plotly.graph_objects as go
    from plotly.subplots import make_subplots

    # Calculate number of rows/columns for subplots
    n_plots = len(rmsd_data) + (1 if control_rmsds is not None else 0)
    n_cols = min(2, n_plots)
    n_rows = (n_plots + n_cols - 1) // n_cols

    # Define different marker symbols for each subplot
    marker_symbols = ['circle', 'diamond', 'square', 'triangle-up', 'star', 'pentagon',
                     'hexagon', 'cross', 'x', 'triangle-down']

    # Create subplots
    fig = make_subplots(
        rows=n_rows, cols=n_cols,
        subplot_titles=['Control' if control_rmsds is not None else ''] + list(rmsd_data.keys()),
        shared_xaxes=True, shared_yaxes=True
    )

    # Set consistent color range for pLDDT
    plddt_min, plddt_max = 50, 100  # typical pLDDT range

    plot_idx = 0  # Start from 0 for marker_symbols indexing
    row = 1
    col = 1

    # Calculate global max RMSD for consistent axis ranges
    max_rmsd = 0
    if control_rmsds is not None:
        max_rmsd = max(max(control_rmsds['ref1_rmsds']),
                      max(control_rmsds['ref2_rmsds'] if reference_pdb2 else control_rmsds['ref1_rmsds']))

    for data in rmsd_data.values():
        max_rmsd = max(max_rmsd,
                      max(data['ref1_rmsds']),
                      max(data['ref2_rmsds'] if reference_pdb2 else data['ref1_rmsds']))

    # Plot control if available
    if control_rmsds is not None:
        plddt_values = []
        for model_name in control_rmsds['model_names']:
            pdb_path = os.path.join(control_path, model_name)
            plddt = extract_plddt_from_pdb(pdb_path)
            plddt_values.append(plddt)

        fig.add_trace(
            go.Scatter(
                x=control_rmsds['ref1_rmsds'],
                y=control_rmsds['ref2_rmsds'] if reference_pdb2 else control_rmsds['ref1_rmsds'],
                mode='markers',
                marker=dict(
                    size=8,
                    symbol=marker_symbols[plot_idx],
                    color=plddt_values,
                    colorscale='Viridis',
                    cmin=plddt_min,
                    cmax=plddt_max,
                    showscale=(plot_idx == 0),
                    colorbar=dict(
                        title='pLDDT',
                        x=1.02,
                        xanchor='left',
                        yanchor='middle',
                        len=1.0
                    )
                ),
                text=[f"Model: {name}<br>pLDDT: {plddt:.2f}"
                      for name, plddt in zip(control_rmsds['model_names'], plddt_values)],
                hoverinfo='text',
                name=f'Control ({marker_symbols[plot_idx]})'
            ),
            row=row, col=col
        )
        plot_idx += 1
        col = col + 1 if col < n_cols else 1
        row = row + 1 if col == 1 else row

    # Plot mutations
    for mutation, data in rmsd_data.items():
        plddt_values = []
        jobname = results[mutation]['jobname']
        for model_name in data['model_names']:
            pdb_path = str(Path(parent_path) / jobname / "out" / "pdbs" / model_name)
            plddt = extract_plddt_from_pdb(pdb_path)
            plddt_values.append(plddt)

        fig.add_trace(
            go.Scatter(
                x=data['ref1_rmsds'],
                y=data['ref2_rmsds'] if reference_pdb2 else data['ref1_rmsds'],
                mode='markers',
                marker=dict(
                    size=8,
                    symbol=marker_symbols[plot_idx % len(marker_symbols)],
                    color=plddt_values,
                    colorscale='Viridis',
                    cmin=plddt_min,
                    cmax=plddt_max,
                    showscale=(plot_idx == 0),
                    colorbar=dict(
                        title='pLDDT',
                        x=1.02,
                        xanchor='left',
                        yanchor='middle',
                        len=1.0
                    )
                ),
                text=[f"Model: {name}<br>pLDDT: {plddt:.2f}<br>RMSD Ref1: {rmsd1:.2f}<br>RMSD Ref2: {rmsd2:.2f}"
                      for name, plddt, rmsd1, rmsd2 in zip(data['model_names'],
                                                          plddt_values,
                                                          data['ref1_rmsds'],
                                                          data['ref2_rmsds'] if reference_pdb2 else data['ref1_rmsds'])],
                hoverinfo='text',
                name=f'{mutation} ({marker_symbols[plot_idx % len(marker_symbols)]})'
            ),
            row=row, col=col
        )
        plot_idx += 1
        col = col + 1 if col < n_cols else 1
        row = row + 1 if col == 1 else row

    # Update layout
    fig.update_layout(
        title_text="RMSD Comparison (colored by pLDDT)",
        showlegend=True,
        height=300*n_rows,
        width=600*n_cols + 100,
        template="plotly_white",
        margin=dict(r=120)
    )

    # Update axes ranges and labels for all subplots
    for i in range(1, n_plots + 1):
        fig.update_xaxes(
            title_text="RMSD vs Reference 1 (Å)",
            range=[0, max_rmsd * 1.1],  # Add 10% padding to max value
            row=(i-1)//n_cols + 1,
            col=(i-1)%n_cols + 1
        )
        fig.update_yaxes(
            title_text="RMSD vs Reference 2 (Å)" if reference_pdb2 else "RMSD vs Reference 1 (Å)",
            range=[0, max_rmsd * 1.1],  # Add 10% padding to max value
            row=(i-1)//n_cols + 1,
            col=(i-1)%n_cols + 1
        )

    fig.update_layout(legend_orientation="h")

    return fig


colorscale_cesar_mpl = [(position, convert_rgb_string_to_tuple(color)) for position, color in colorscale_cesar]
cesar_cmap = LinearSegmentedColormap.from_list('cesar', colorscale_cesar_mpl)

def create_output_dirs():
    """Create directory structure for outputs."""
    base_dir = 'rmsd_analysis'
    dirs = {
        'base': base_dir,
        'landscapes': os.path.join(base_dir, 'landscapes'),
        'distributions': os.path.join(base_dir, 'distributions'),
        'combined': os.path.join(base_dir, 'combined'),
        'data': os.path.join(base_dir, 'data')
    }

    for dir_path in dirs.values():
        os.makedirs(dir_path, exist_ok=True)

    return dirs

output_dirs = create_output_dirs()

# Initialize analysis
print("Initializing RMSD Analysis...")
print(f"Analysis Parameters:")
print(f"- Atom selection: {atom_selection}")
print(f"- Region: {f'residues {start_residue}-{end_residue}' if use_region else 'entire structure'}")
print(f"- Control path: {control_path}")
print("-" * 50)

try:
    # Load reference structures
    ref_coords1 = extract_coordinates(reference_pdb1, atom_selection, start_residue, end_residue)
    print(f"Loaded reference structure 1: {reference_pdb1}")
    print(f"Number of atoms used: {len(ref_coords1)}")

    if reference_pdb2:
        ref_coords2 = extract_coordinates(reference_pdb2, atom_selection, start_residue, end_residue)
        print(f"Loaded reference structure 2: {reference_pdb2}")
        print(f"Number of atoms used: {len(ref_coords2)}")
    else:
        ref_coords2 = None

    # Calculate control RMSDs if requested
    if include_control:
        print("\nProcessing control structures...")
        control_rmsds = calculate_model_rmsds(control_path, ref_coords1, ref_coords2)
        control_stats = calculate_statistics(control_rmsds)
        print("\nControl Statistics:")
        for ref, stats in control_stats.items():
            print(f"\n{ref.upper()} Statistics:")
            for key, value in stats.items():
                print(f"- {key}: {value:.3f}" if isinstance(value, float) else f"- {key}: {value}")

except Exception as e:
    print(f"Error during initialization: {str(e)}")
    raise

# Main Execution
try:
    # Calculate RMSD for all mutants
    print("\nProcessing mutant structures...")
    rmsd_results = {}

    for mutation, data in tqdm(results.items(), desc="Processing mutations"):
        if data['status'] == 'completed':
            jobname = data['jobname'][0] # THis is not correct bc 'jobname' is now a tuple
            model_pdbs = find_model_pdbs(jobname)

            if model_pdbs:
                print(f"\nProcessing {mutation}:")
                rmsd_results[mutation] = {
                    'ref1_rmsds': [],
                    'ref2_rmsds': [] if reference_pdb2 else None,
                    'model_names': []
                }

                for pdb_file in model_pdbs:
                    try:
                        model_coords = extract_coordinates(str(pdb_file), atom_selection,
                                                        start_residue, end_residue)

                        # Calculate RMSD against reference 1
                        rmsd1 = float(jnp_rmsd(jnp.array(ref_coords1), jnp.array(model_coords)))
                        rmsd_results[mutation]['ref1_rmsds'].append(rmsd1)

                        # Calculate RMSD against reference 2 if provided
                        if reference_pdb2:
                            rmsd2 = float(jnp_rmsd(jnp.array(ref_coords2), jnp.array(model_coords)))
                            rmsd_results[mutation]['ref2_rmsds'].append(rmsd2)

                        rmsd_results[mutation]['model_names'].append(pdb_file.name)
                    except Exception as e:
                        print(f"Warning: Could not process {pdb_file.name}: {str(e)}")

                print(f"Successfully processed {len(rmsd_results[mutation]['ref1_rmsds'])} models")

    # Create visualizations
    print("\nGenerating visualizations...")

    if show_violin:
        print("\n📊 Creating violin plots...")
        violin_fig = plot_violin_distributions(rmsd_results, include_control)
        violin_fig.savefig(os.path.join(output_dirs['distributions'], 'rmsd_distributions.png'), dpi=300)
        plt.close(violin_fig)

    if show_landscapes:
        print("\n🗺️ Creating landscape plots...")
        # Individual landscapes
        for mutation, data in rmsd_results.items():
            fig = plot_rmsd_landscape(
                data['ref1_rmsds'],
                data['ref2_rmsds'] if reference_pdb2 else data['ref1_rmsds'],
                f"{mutation} RMSD Landscape"
            )
            fig.savefig(os.path.join(output_dirs['landscapes'], f'{mutation}_landscape.png'), dpi=300)
            plt.close(fig)

        # Control landscape if included
        if include_control:
            control_fig = plot_rmsd_landscape(
                control_rmsds['ref1_rmsds'],
                control_rmsds['ref2_rmsds'] if reference_pdb2 else control_rmsds['ref1_rmsds'],
                "Control RMSD Landscape"
            )
            control_fig.savefig(os.path.join(output_dirs['landscapes'], 'control_landscape.png'), dpi=300)
            plt.close(control_fig)

        # Summary landscape
        summary_fig = create_landscape_summary(rmsd_results, control_rmsds if include_control else None)
        summary_fig.savefig(os.path.join(output_dirs['combined'], 'landscape_summary.png'), dpi=300)
        plt.close(summary_fig)

    if show_scatterplot:
        print("\n📊 Creating interactive plots...")
        interactive_fig = create_interactive_plots(rmsd_results, control_rmsds if include_control else None)

        # Save interactive plot
        interactive_fig.write_html(os.path.join(output_dirs['data'], 'interactive_rmsd_plots.html'))


    # Generate statistics and save data
    print("\n📈 Generating statistics...")
    stats_df = create_summary_statistics(rmsd_results, control_rmsds if include_control else None)

    # Save all data
    print("\n💾 Saving results...")
    stats_df.to_csv(os.path.join(output_dirs['data'], 'rmsd_statistics.csv'), index=False)

    # Save raw RMSD data
    raw_data = {
        'control': control_rmsds if include_control else None,
        'mutations': rmsd_results
    }
    np.save(os.path.join(output_dirs['data'], 'raw_rmsd_data.npy'), raw_data)

    # Display summary table
    print("\n📋 RMSD Analysis Summary:")
    data_table.DataTable(stats_df, include_index=False, num_rows_per_page=20)

    # Display some key findings
    print("\n🔍 Key Findings:")
    if include_control:
        print("\nControl Structure:")
        print(f"- Mean RMSD vs Ref1: {control_stats['ref1']['mean']:.3f} Å")
        if reference_pdb2:
            print(f"- Mean RMSD vs Ref2: {control_stats['ref2']['mean']:.3f} Å")

    print("\nMutant Structures:")
    for mutation, data in rmsd_results.items():
        stats = calculate_statistics(data)
        print(f"\n{mutation}:")
        print(f"- Mean RMSD vs Ref1: {stats['ref1']['mean']:.3f} Å")
        if reference_pdb2:
            print(f"- Mean RMSD vs Ref2: {stats['ref2']['mean']:.3f} Å")



    # Save detailed data
    print("\n💾 Saving detailed results...")
    detailed_df = save_detailed_data(rmsd_results,
                                   control_rmsds if include_control else None,
                                   output_dirs['data'])

    # Save summary statistics
    stats_df.to_csv(os.path.join(output_dirs['data'], 'rmsd_summary_statistics.csv'),
                    index=False)

    # Save directory structure
    if 'base_save_dir' in locals():
        # Save unzipped directory
        unzipped_dir = os.path.join(base_save_dir, 'rmsd_analysis_unzipped')
        if os.path.exists(unzipped_dir):
            shutil.rmtree(unzipped_dir)
        shutil.copytree(output_dirs['base'], unzipped_dir)
        print(f"\nUnzipped analysis saved to: {unzipped_dir}")

        # Create zip file
        zip_filename = os.path.join(base_save_dir, 'rmsd_analysis.zip')
        if os.path.exists(zip_filename):
            os.remove(zip_filename)
        with zipfile.ZipFile(zip_filename, 'w', zipfile.ZIP_DEFLATED) as zipf:
            for root, _, files in os.walk(output_dirs['base']):
                for file in files:
                    file_path = os.path.join(root, file)
                    arcname = os.path.relpath(file_path, output_dirs['base'])
                    zipf.write(file_path, arcname)
        print(f"Zipped analysis saved to: {zip_filename}")

except Exception as e:
    print(f"\n❌ Error during analysis: {str(e)}")
    print("\nDebug information:")
    print(f"Reference PDB 1: {reference_pdb1}")
    print(f"Reference PDB 2: {reference_pdb2}")
    print(f"Parent path: {parent_path}")
    print(f"Atom selection: {atom_selection}")
    print(f"Region: {start_residue}-{end_residue}" if use_region else "entire structure")
    raise

print("\n✅ Analysis complete!")

# Return results for further use if needed
analysis_results = {
    'rmsd_results': rmsd_results,
    'control_results': control_rmsds if include_control else None,
    'statistics': stats_df,
    'parameters': {
        'atom_selection': atom_selection,
        'region': (start_residue, end_residue) if use_region else None,
        'references': [reference_pdb1, reference_pdb2 if reference_pdb2 else None]
    }
}

print("\n📋 Detailed RMSD Data Preview:")
data_table.DataTable(detailed_df.head(20), include_index=False)

In [ ]:
# Display in notebook
interactive_fig.show()

# FAQ and Usage Guide

In [ ]:
# @markdown ## What's New in This Version
# @markdown - Added mutation capability with `iterative_single_mask_mutate` strategy
# @markdown - Improved MSA handling with multiple methods
# @markdown - Automatic results compression and download
# @markdown - Enhanced memory management
# @markdown - More flexible position selection

# @markdown ## Masking Strategies Explained

# @markdown ### 1. Iterative Single Position (iterative_single)
# @markdown - **What it does**: Masks one position at a time, runs AF2 for each position
# @markdown - **Usage options**:
# @markdown   - Leave "Positions to Process" empty → analyze entire sequence
# @markdown   - Enter specific positions (e.g., "1,5,10") → analyze only those positions
# @markdown - **Example**: With "1,5,10" → three predictions, each masking one position
# @markdown - **Use case**: Understanding individual residue contributions

# @markdown ### 2. Iterative Single Position with Mutation (iterative_single_mask_mutate)
# @markdown - **What it does**: Combines masking and mutation at each position
# @markdown - **Usage**:
# @markdown   - Enter positions to analyze
# @markdown   - Specify mutation (e.g., "S89R")
# @markdown - **Example**: Position "89", mutation "S89R" → predicts structure with position 89 masked and mutated
# @markdown - **Use case**: Studying mutation effects in masked contexts

# @markdown ### 3. Mask Positions (mask_positions)
# @markdown - **What it does**: Masks all specified positions simultaneously
# @markdown - **Usage**: Enter all positions to mask (e.g., "1,5,10")
# @markdown - **Example**: "1,5,10" → one prediction with all three positions masked
# @markdown - **Use case**: Analyzing combined effects of multiple positions

# @markdown ### 4. Unmask Positions (unmask_positions)
# @markdown - **What it does**: Masks everything EXCEPT specified positions
# @markdown - **Usage**: Enter positions to keep unmasked
# @markdown - **Example**: "1,5,10" in 100-residue protein → masks positions 2-4,6-9,11-100
# @markdown - **Use case**: Focusing on specific structural regions

# @markdown ## Configuration Guide

# @markdown ### Basic Parameters
# @markdown ```python
# @markdown sequence = "YOUR_SEQUENCE"  # Input protein sequence
# @markdown jobname_prefix = "experiment1"  # Prefix for output files
# @markdown masking_strategy = "iterative_single"  # Choose strategy
# @markdown positions_str = "1,5,10"  # Positions to process (optional)
# @markdown ```

# @markdown ### Advanced Parameters
# @markdown ```python
# @markdown num_recycles = 6  # Number of recycles (3-20)
# @markdown num_seeds = 6  # Number of seeds (3-20)
# @markdown msa_method = "mmseqs2"  # MSA generation method
# @markdown run_control = True  # Run unmasked prediction
# @markdown ```

# @markdown ## Common Questions

# @markdown ### Q: How do I set up my first experiment?
# @markdown A: Basic setup steps:
# @markdown 1. Paste your sequence
# @markdown 2. Choose masking strategy
# @markdown 3. Set positions (if needed)
# @markdown 4. Run the cell
# @markdown Default parameters work well for most cases

# @markdown ### Q: Which MSA method should I use?
# @markdown A: Available options:
# @markdown - `mmseqs2`: Best for most cases
# @markdown - `single_sequence`: Quick testing
# @markdown - `custom_a3m`: Use existing alignment
# @markdown - `custom_fas`: Use FASTA file
# @markdown - `custom_sto`: Use Stockholm file

# @markdown ### Q: How long will it take?
# @markdown A: Approximate times per prediction:
# @markdown - Basic run (3 recycles): ~15-20 minutes
# @markdown - Standard run (6 recycles): ~25-30 minutes
# @markdown - High-accuracy (12 recycles): ~45-60 minutes
# @markdown - Total time = (predictions × time per prediction)

# @markdown ### Q: Memory issues?
# @markdown A: Try these solutions:
# @markdown ```python
# @markdown num_recycles = 3  # Reduce recycles
# @markdown num_seeds = 3  # Reduce seeds
# @markdown msa_method = "single_sequence"  # Simpler MSA
# @markdown ```

# @markdown ### Q: Recommended settings for different scenarios?
# @markdown A: Common configurations:
# @markdown ```python
# @markdown # Quick test
# @markdown num_recycles = 3
# @markdown num_seeds = 3
# @markdown msa_method = "single_sequence"
# @markdown
# @markdown # Standard run
# @markdown num_recycles = 6
# @markdown num_seeds = 6
# @markdown msa_method = "mmseqs2"
# @markdown
# @markdown # High accuracy
# @markdown num_recycles = 12
# @markdown num_seeds = 12
# @markdown msa_method = "mmseqs2"
# @markdown ```

# @markdown ### Q: Where are my results?
# @markdown A: Results are:
# @markdown 1. Saved in your specified parent_path
# @markdown 2. Automatically zipped
# @markdown 3. Downloaded to your computer
# @markdown 4. Named: [jobname_prefix]_[strategy]_[positions].zip

# @markdown ### Q: Should I run control predictions?
# @markdown A: Recommended when:
# @markdown - First time running sequence
# @markdown - Comparing masked vs unmasked
# @markdown - Validating mutations
# @markdown Skip if you already have unmasked prediction

# @markdown ## Mutation Analysis Guide

# @markdown ### Example mutation workflow:
# @markdown ```python
# @markdown # Single mutation
# @markdown masking_strategy = "iterative_single_mask_mutate"
# @markdown positions_str = "89"
# @markdown mutations = "S89R"
# @markdown
# @markdown # Run with validation
# @markdown run_control = True  # Compare with wild-type
# @markdown ```

# @markdown ### Tips for mutation analysis:
# @markdown - Always run control for comparison
# @markdown - Use consistent parameters between runs
# @markdown - Consider running multiple seeds
# @markdown - Check surrounding residues

# @markdown ## Advanced Usage

# @markdown ### Example: Regional analysis
# @markdown ```python
# @markdown # Analyze binding site (positions 10-20)
# @markdown masking_strategy = "unmask_positions"
# @markdown positions_str = "10,11,12,13,14,15,16,17,18,19,20"
# @markdown ```

# @markdown ### Example: Multiple positions
# @markdown ```python
# @markdown # Mask multiple specific positions
# @markdown masking_strategy = "mask_positions"
# @markdown positions_str = "1,5,10,15,20"
# @markdown ```

# @markdown ### Example: High-throughput scanning
# @markdown ```python
# @markdown # Scan entire sequence
# @markdown masking_strategy = "iterative_single"
# @markdown positions_str = ""  # Empty for full sequence
# @markdown ```